# Attribute interference — Colab runner
Run cells in order on an NVIDIA GPU runtime. This notebook calls the repository CLI; models are loaded in separate processes. No training or fine-tuning. Qwen is not quantized.

First upload this notebook to Colab, choose **Runtime → Change runtime type → GPU**, and obtain the project either from the team Git repository or a ZIP. Full baseline: **576 prompts × four seeds = 2,304 images**. GPU runtime and checkpoint compatibility must pass the smoke test before a full run.


In [ ]:
from pathlib import Path
import subprocess, sys, os
REPO_URL = ""  # Fill with the actual team Git URL; leave blank to upload a project ZIP.
REPO = Path("/content/cv_project")
if not (REPO / "pyproject.toml").exists():
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
    else:
        from google.colab import files
        import zipfile, io
        uploaded = files.upload()
        zips = [data for name, data in uploaded.items() if name.endswith(".zip")]
        if len(zips) != 1:
            raise ValueError("Upload exactly one project ZIP")
        REPO.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(zips[0])) as archive:
            # Accept either a flat source archive or one top-level project directory.
            members = [i for i in archive.infolist() if not i.is_dir() and "__MACOSX" not in i.filename]
            manifests = [i.filename for i in members if i.filename.endswith("pyproject.toml")]
            if len(manifests) != 1:
                raise ValueError("ZIP must contain one project pyproject.toml")
            prefix = manifests[0][:-len("pyproject.toml")]
            for info in members:
                if not info.filename.startswith(prefix):
                    continue
                rel = Path(info.filename[len(prefix):])
                if rel.is_absolute() or ".." in rel.parts:
                    raise ValueError("Unsafe ZIP member")
                if any(p in {".venv", ".git", "__pycache__", "outputs"} for p in rel.parts):
                    continue
                target = REPO / rel
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(info))
os.chdir(REPO)
print("Repository:", REPO)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."], check=True)
# Colab may contain unrelated preinstalled packages with conflicting pins.
# Print those conflicts, then verify our own imports in the next cell.
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)


If Torch/Transformers were already imported, restart the session now and rerun the repository-selection cell. Do not reinstall on every resume unless necessary. Existing runs require the recorded environment.


In [ ]:
import torch
from diffusers import StableDiffusionPipeline, StableDiffusionXLPipeline
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, Owlv2ForObjectDetection, Owlv2Processor
assert torch.cuda.is_available(), "Select a CUDA GPU runtime"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")
from huggingface_hub import model_info
for model in ["stable-diffusion-v1-5/stable-diffusion-v1-5", "stabilityai/stable-diffusion-xl-base-1.0",
              "Qwen/Qwen2.5-VL-7B-Instruct", "google/owlv2-base-patch16-ensemble"]:
    print(model, model_info(model).sha)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUTPUTS = Path("/content/drive/MyDrive/attribute_interference/v1")
OUTPUTS.mkdir(parents=True, exist_ok=True)
def run(command, config, name, *extra):
    subprocess.run([sys.executable, "-m", "attribute_interference.cli", command,
                    "--config", str(REPO / "configs" / (config + ".yaml")),
                    "--output", str(OUTPUTS / name), *map(str, extra)], check=True, cwd=REPO)
print("Persistent outputs:", OUTPUTS)


## 1. Technical mini-set — eight prompts per model
Inspect outputs and retry the same cells to verify resume. A successful smoke run checks integration, not evaluator accuracy. GPU OOM requires sufficient RAM/offload or a larger GPU; do not quantize Qwen.


In [ ]:
for config, name in [("smoke", "smoke_sd15"), ("smoke_sdxl", "smoke_sdxl")]:
    for command in ["prepare", "generate", "evaluate", "analyze"]:
        run(command, config, name)


## 2. SDXL development pilot — 160 images
Inspect the prompt assignments before generation. Use only development data for changes to questions, parsing and detector thresholds. Source/config changes require a new output directory.


In [ ]:
run("prepare", "pilot_sdxl", "pilot_sdxl")
from attribute_interference.storage import read_csv
read_csv(OUTPUTS / "pilot_sdxl/prompts.csv")[:12]


In [ ]:
run("generate", "pilot_sdxl", "pilot_sdxl")
run("evaluate", "pilot_sdxl", "pilot_sdxl", "--backend", "qwen")
run("evaluate", "pilot_sdxl", "pilot_sdxl", "--backend", "owl")
run("analyze", "pilot_sdxl", "pilot_sdxl")


In [ ]:
run("export-annotations", "pilot_sdxl", "pilot_sdxl")
# Display the selected images in Colab; the gallery HTML is also saved in Drive.
from IPython.display import display, Image as DisplayImage
review = read_csv(OUTPUTS / "pilot_sdxl/review/annotations.csv")
for row in review:
    print(row["image_id"], row["prompt"])
    display(DisplayImage(filename=str(OUTPUTS / "pilot_sdxl/images" / (row["image_id"] + ".png")), width=384))


**Manual step:** fill `pilot_sdxl/review/annotations.csv` in Drive: reviewer name plus `true`, `false`, or `unknown` for object presence and each active factor. Leave inactive factors blank. If the object is absent, all active factors are false. Review singleton-factor accuracy against the proposal's 60% criterion. Tune only on development data and repeat review in a new run if needed. The export command protects existing human edits.

Freezing does not assert high agreement; inspect the saved agreement counts. Continue to main evaluation only after the team accepts the pilot settings.


In [ ]:
run("freeze-evaluator", "pilot_sdxl", "pilot_sdxl")
import json
FROZEN = OUTPUTS / "pilot_sdxl/evaluator_freeze.json"
print(json.dumps(json.loads(FROZEN.read_text()), indent=2))


## 3. Full SD 1.5 baseline — four seeds, 2,304 images
Run these stages separately; resume any interrupted stage by rerunning the same cell. No change to source/config/output name during a run. Generation may proceed independently, but main evaluation requires the reviewed pilot freeze.


In [ ]:
run("prepare", "main_sd15", "main_sd15")
run("generate", "main_sd15", "main_sd15")


In [ ]:
FROZEN = OUTPUTS / "pilot_sdxl/evaluator_freeze.json"
run("evaluate", "main_sd15", "main_sd15", "--backend", "qwen", "--frozen", FROZEN)
run("evaluate", "main_sd15", "main_sd15", "--backend", "owl", "--frozen", FROZEN)


In [ ]:
run("analyze", "main_sd15", "main_sd15")
import json
summary = json.loads((OUTPUTS / "main_sd15/analysis/summary.json").read_text())
print(json.dumps(summary["run"], indent=2))
display(read_csv(OUTPUTS / "main_sd15/analysis/attributes.csv"))
for figure in ["attribute_accuracy.png", "complexity.png", "interference.png"]:
    display(DisplayImage(filename=str(OUTPUTS / "main_sd15/analysis" / figure)))
assert summary["run"]["complete_generation"], "Baseline generation is incomplete; inspect errors and resume."
assert summary["run"]["complete_scoring"], "Scoring coverage is incomplete; inspect raw answers/errors."


## 4. Optional next experiment — full SDXL comparison
Disabled by default. Use identical main prompt assignments, four seeds and the same frozen evaluator. Keep SDXL outputs separate. The first submission does not require this second full run.


In [ ]:
RUN_FULL_SDXL = False
if RUN_FULL_SDXL:
    run("prepare", "main_sdxl", "main_sdxl")
    run("generate", "main_sdxl", "main_sdxl")
    run("evaluate", "main_sdxl", "main_sdxl", "--frozen", OUTPUTS / "pilot_sdxl/evaluator_freeze.json")
    run("analyze", "main_sdxl", "main_sdxl")


## Share results
Share the real team Git URL in the report. Copy reviewed compact CSV/PNG summaries into the repository only after the real run; keep the image collection in Drive. Cite model revisions, actual hardware, coverage, limitations and measured timings from the run files. Never report synthetic unit-test fixtures as results.
